# Auto-sklearn2


In [1]:
!pip -q install -U auto-sklearn2

In [2]:
import os
import pandas as pd

path = 'bank-full.csv'
if not os.path.exists(path):
    alt_path = '/mnt/data/bank-full.csv'
    if os.path.exists(alt_path):
        path = alt_path

df = pd.read_csv(path, sep=';')
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [3]:
label = 'y'
y = df[label].map({'no': 0, 'yes': 1})
X = df.drop(columns=[label])

y.value_counts()


,count
y,
0,39922
1,5289


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train.shape, X_test.shape


((36168, 16), (9043, 16))

## Preprocesado ráṕido (One-Hot para categóricas)

`auto-sklearn2` (igual que `auto-sklearn`) trabaja con matrices numéricas; aplicamos `OneHotEncoder` para categóricas.


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = [c for c in X_train.columns if c not in cat_cols]

preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', 'passthrough', num_cols),
    ],
    remainder='drop'
)

len(cat_cols), len(num_cols)


(9, 7)

## Entrenamiento con auto-sklearn2

In [6]:
from auto_sklearn2 import AutoSklearnClassifier
from sklearn.pipeline import Pipeline
import inspect

sig = inspect.signature(AutoSklearnClassifier)
params = set(sig.parameters.keys())

kwargs = {}
if 'time_left_for_this_task' in params:
    kwargs['time_left_for_this_task'] = 60
if 'per_run_time_limit' in params:
    kwargs['per_run_time_limit'] = 10
if 'n_jobs' in params:
    kwargs['n_jobs'] = -1
elif 'random_state' in params:
    kwargs['random_state'] = 42

automl = AutoSklearnClassifier(**kwargs)

model = Pipeline(steps=[
    ('preprocess', preprocess),
    ('automl', automl)
])

model.fit(X_train, y_train)


Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['job', 'marital',
                                                   'education', 'default',
                                                   'housing', 'loan', 'contact',
                                                   'month', 'poutcome']),
                                                 ('num', 'passthrough',
                                                  ['age', 'balance', 'day',
                                                   'duration', 'campaign',
                                                   'pdays', 'previous'])])),
                ('automl', AutoSklearnClassifier())])

## Evaluación (ROC AUC)


In [7]:
import numpy as np
from sklearn.metrics import roc_auc_score, classification_report

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

if hasattr(model, 'predict_proba'):
    proba = model.predict_proba(X_test)[:, 1]
elif hasattr(model, 'decision_function'):
    scores = model.decision_function(X_test)
    proba = sigmoid(scores)
else:
    # Fallback: usar predicción de clase (no ideal para AUC)
    proba = model.predict(X_test)

auc = roc_auc_score(y_test, proba)
print('ROC AUC:', auc)

pred = (proba >= 0.5).astype(int)
print(classification_report(y_test, pred, digits=4))


ROC AUC: 0.9240266780932584
              precision    recall  f1-score   support

           0     0.9255    0.9711    0.9477      7985
           1     0.6526    0.4102    0.5038      1058

    accuracy                         0.9055      9043
   macro avg     0.7891    0.6906    0.7258      9043
weighted avg     0.8936    0.9055    0.8958      9043



/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


## Inspección del modelo encontrado


In [8]:
automl_est = model.named_steps['automl']

for attr in ['sprint_statistics', 'show_models', 'leaderboard']:
    if hasattr(automl_est, attr):
        print(f"\n--- {attr}() ---")
        try:
            out = getattr(automl_est, attr)()
            print(out)
        except TypeError:
            try:
                out = getattr(automl_est, attr)(detailed=True)
                print(out)
            except Exception as e:
                print('No se pudo ejecutar:', e)
        except Exception as e:
            print('No se pudo ejecutar:', e)


## Guardar y cargar el modelo (opcional)


In [9]:
import joblib
joblib.dump(model, 'autosklearn2_bank_model.joblib')
print('Guardado en autosklearn2_bank_model.joblib')


Guardado en autosklearn2_bank_model.joblib


---
### Podemos probar...
1. Subir `time_left_for_this_task` a 600s y comparar AUC.
2. Probar diferentes umbrales (0.3, 0.4, 0.5) y medir recall de la clase positiva.
3. Comparar con otro modelo: `LogisticRegression` usando el mismo `preprocess`.
